last modified date : 2026.03.15  
제작 : 박광석 (모두의연구소)

# 랭체인으로 RAG 시작하기

해당 노트는 Langchain으로 RAG를 구현하기 위해 필요한
각 컴포넌트인 Document Loaders, Text splitters, Text embeddings, Vectorstores, Retriever를 다룹니다  




### Step 0 : 설치와 준비  
Langchain 설치 및 Gemini API 키를 등록하도록 합니다.  

In [8]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [9]:
!pip install -U -q langchain langchain-openai
!pip install -U -q langchain-community langchain-core
!pip install -U langchain-text-splitters


In [10]:
import os


In [11]:
from google.colab import userdata
os.environ['OPENAI_API_KEY'] = userdata.get('OPENAI_KEY')

In [12]:
#! curl ipinfo.io

In [13]:
from langchain_openai import ChatOpenAI

# OpenAI API를 사용하는 설정으로 변경
# 모델명은 필요에 따라 "gpt-4o", "gpt-4-turbo", "gpt-3.5-turbo" 등으로 바꿀 수 있습니다.
llm = ChatOpenAI(
    model="gpt-4o",
    temperature=0.0,
)

In [14]:
!pip install -q pypdf pdf2image docx2txt pdfminer unstructured #의존성 모듈을 설치합니다

### Step 1 : Document Loaders 사용해보기  

Document Loader는 다양한 형태의 원본 데이터를  
LLM이 이해할 수 있는 Document 객체(text + metadata) 로 변환하는 역할을 합니다.

PDF, 웹페이지, CSV와 같이 형식이 서로 다른 문서들을 일관된 구조로 파싱하여, 이후 Chunking·Embedding·검색(Retrieval) 단계에서
바로 사용할 수 있도록 만들어줍니다.

즉, Document Loader는
**RAG 파이프라인의 가장 첫 단계에서 “데이터를 읽을 수 있는 형태로 정리하는 역할을 담당**합니다.

공식 문서에서는 지원되는 다양한 Loader 목록을 확인할 수 있습니다.
https://python.langchain.com/docs/modules/data_connection/document_loaders/

#### PDFLoader 사용  
이번 실습에서는 가장 많이 사용되는 문서 형식인 PDF 파일을 대상으로
PyPDFLoader를 사용해 문서를 불러옵니다.

실습을 위해, 질의응답에 활용하고 싶은 PDF 파일을 먼저 Colab 환경(또는 Drive)에 업로드해주세요.

PDFLoader는 각 페이지를 하나의 Document 단위로 변환하며,
이 단계에서 생성된 문서들은 이후 Text Splitter를 통해 의미 단위로 다시 분할됩니다.

In [15]:
from langchain_community.document_loaders import PyPDFLoader

loader = PyPDFLoader("/content/Demian.pdf")
pages = loader.load_and_split()

/tmp/ipykernel_11321/128205925.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


In [16]:
pages[0]

Document(metadata={'producer': 'Adobe Acrobat Standard DC 19 Paper Capture Plug-in', 'creator': 'ScanFix(TM) Enhanced', 'creationdate': '2015-09-10T01:40:29+00:00', 'moddate': '2019-01-30T17:47:47+01:00', 'source': '/content/Demian.pdf', 'total_pages': 182, 'page': 0, 'page_label': '1'}, page_content='DEMIAN \n• \nDownloaded from https://www.holybooks.com')

In [17]:
print(pages[10])

page_content='TWO WOR.LDS 
Finally, out of sheer nervousness, I began to talk. I 
invented a long story of robbery, in which I featured as 
the hero. One night in the comer by the mill a friend 
and I ha.d stolen a whole sackful of apples, not just 
ordinary apples but pippins, golden pippins of the best 
kind at that. I was taking refuge in my story from the 
dangers of the moment and found no difficulty in invent­
ing and relating it. In order not to dry up too soon and 
perhaps become involved in something worse, I gave full 
rein to my narrative powers. One of us, I reported, had 
always stood guard while the other sat in the tree and 
chucked the apples down, and the sack had got so heavy 
that in the end we had to open it and leave half behind, 
but we came back half an hour later and fetched them 
too. 
I hoped for some applause at the end of my story; I 
had warmed up to the narrative aJ: last, carried away by 
my own eloquence. The two smaller boys were silent, 
waiting, Lut F

출력 결과를 보기 쉽게 확인하기 위해,
Document 객체 전체가 아닌 실제 텍스트 본문이 담긴 page_content만 선택하여 확인해보겠습니다.

In [18]:
print(pages[10].page_content)

TWO WOR.LDS 
Finally, out of sheer nervousness, I began to talk. I 
invented a long story of robbery, in which I featured as 
the hero. One night in the comer by the mill a friend 
and I ha.d stolen a whole sackful of apples, not just 
ordinary apples but pippins, golden pippins of the best 
kind at that. I was taking refuge in my story from the 
dangers of the moment and found no difficulty in invent­
ing and relating it. In order not to dry up too soon and 
perhaps become involved in something worse, I gave full 
rein to my narrative powers. One of us, I reported, had 
always stood guard while the other sat in the tree and 
chucked the apples down, and the sack had got so heavy 
that in the end we had to open it and leave half behind, 
but we came back half an hour later and fetched them 
too. 
I hoped for some applause at the end of my story; I 
had warmed up to the narrative aJ: last, carried away by 
my own eloquence. The two smaller boys were silent, 
waiting, Lut Franz Kromer ga

#### CSVLoader

SV 파일은 행(row) 단위로 구조화된 데이터를 담고 있는 형식으로,
LangChain의 CSVLoader를 사용하면 각 행을 하나의 Document 객체로 변환할 수 있습니다.

이렇게 변환된 문서들은 이후 PDF나 웹 문서와 동일하게
Embedding, VectorStore, Retrieval 단계에서 함께 활용할 수 있습니다.

실습을 위해, CSV 파일을 먼저 Colab 환경(또는 Drive)에 업로드해주세요.

In [19]:
from langchain_community.document_loaders import CSVLoader

loader = CSVLoader("/content/titanic.csv")

data = loader.load()

In [20]:
data[:3]

[Document(metadata={'source': '/content/titanic.csv', 'row': 0}, page_content='PassengerId: 1\nSurvived: 0\nPclass: 3\nName: Braund, Mr. Owen Harris\nSex: male\nAge: 22\nSibSp: 1\nParch: 0\nTicket: A/5 21171\nFare: 7.25\nCabin: \nEmbarked: S'),
 Document(metadata={'source': '/content/titanic.csv', 'row': 1}, page_content='PassengerId: 2\nSurvived: 1\nPclass: 1\nName: Cumings, Mrs. John Bradley (Florence Briggs Thayer)\nSex: female\nAge: 38\nSibSp: 1\nParch: 0\nTicket: PC 17599\nFare: 71.2833\nCabin: C85\nEmbarked: C'),
 Document(metadata={'source': '/content/titanic.csv', 'row': 2}, page_content='PassengerId: 3\nSurvived: 1\nPclass: 3\nName: Heikkinen, Miss. Laina\nSex: female\nAge: 26\nSibSp: 0\nParch: 0\nTicket: STON/O2. 3101282\nFare: 7.925\nCabin: \nEmbarked: S')]

#### 웹베이스로더  
웹베이스 로더는 웹페이지에 포함된 텍스트 콘텐츠를 직접 파싱하여 Document 객체로 변환하는 역할을 합니다.  
이를 통해 뉴스 기사, 블로그 글, 공지사항과 같은 실시간으로 업데이트되는 웹 문서를 RAG 시스템의 지식 소스로 활용할 수 있습니다.  
이번 실습에서는 실제 뉴스 기사를 예제로 사용하여,
웹페이지의 내용을 불러오고 텍스트 형태로 변환하는 과정을 살펴봅니다.  

실습에 사용할 웹페이지는 다음과 같습니다.  
https://it.chosun.com/news/articleView.html?idxno=2023092111831

In [21]:
from langchain_community.document_loaders import WebBaseLoader

In [22]:
loader = WebBaseLoader("https://it.chosun.com/news/articleView.html?idxno=2023092111831")
documents = loader.load()

print(documents[0].page_content)





































모두의연구소 ‘AI학교 아이펠’, ICLR 2024 워크숍 혁신 기술 논문 채택 < 일반 < 기업 < 기사본문 - IT조선











 






















































주요서비스 바로가기
본문 바로가기
매체정보 바로가기
로그인 바로가기
기사검색 바로가기
전체서비스 바로가기




























 






전체메뉴



기사검색








기사검색


검색

닫기














기사검색









기사검색


검색

닫기











로그인




facebook

post
youtube



UPDATED. 2026-05-26 15:02 (화) 




기사검색









기업

일반
모바일·가전
방송·통신
반도체·디스플레이
SW·보안
중공업·에너지
소부장·스타트업
유통·쇼핑
프롭테크·부동산



모빌리티

자동차·모빌리티
로봇·드론·항공
방산



게임·콘텐츠

게임·인터넷
메타버스·VR
키덜트
미디어·엔터



과학·헬스

과학·우주
의학·정책
제약·바이오



파이낸스

금융
증권
핀테크·블록체인



칼럼·인터뷰

칼럼
기고
인터뷰



알림

알립니다
인사
부음
보도자료



컴퓨팅·AI

일반
코딩
에듀테크
테크리포트



GLOBAL


GLOBAL
기획·연재
속보
백과사전











속보




기업


일반


모바일·가전


방송·통신


반도체·디스플레이


SW·보안


중공업·에너지


소부장·스타트업


유통·쇼핑


프롭테크·부동산




모빌리티


자동차·모빌리티


로봇·드론·항공


방산




게임·콘텐츠


게임·인터넷


메타버스·VR


키덜트


미디어·엔터




과학·헬스


과학·우주


의학·정책


제약·바이오




파이낸스


금융


증권


핀테크·블록체인




칼럼·인터뷰


칼럼


기고

주석을 해제하고 코드를 실행하면,
해당 웹페이지에 포함된 본문 텍스트 전체를 불러와 확인할 수 있습니다.  

웹페이지, PDF, CSV 등 서로 다른 형식의 문서들이
모두 텍스트 형태로 정상적으로 파싱된 것을 확인할 수 있습니다.  

이제 이 텍스트를 **전처리(불필요한 요소 제거, 정제)** 한 뒤,
Chunking과 Embedding 단계에 활용할 수 있습니다.  

### Step2 : TextSplitters 사용해보기  
Text Splitter는 긴 텍스트 문서를 **의미를 유지한 작은 단위(Chunk)** 로 분할하는 역할을 합니다.  
LLM은 한 번에 처리할 수 있는 토큰 수에 제한이 있기 때문에, 문서를 그대로 입력하는 대신 Splitter를 통해 분할된 여러 Chunk를 입력받아 처리하게 됩니다.  

이 과정을 통해 긴 문서에서도 토큰 길이 제약을 극복하고, 필요한 부분만 효율적으로 검색할 수 있습니다.  

분할된 각 Chunk는 이후 단계에서 1:1로 Embedding되어 VectorStore에 저장되며,
이 Chunk 단위가 RAG 시스템에서 검색과 응답의 기본 단위가 됩니다.  

In [23]:
from langchain_text_splitters import CharacterTextSplitter, RecursiveCharacterTextSplitter

CharacterTextSplitter는
하나의 고정된 구분자(separator)를 기준으로 텍스트를 분할하는 방식입니다.
구현이 단순하고 직관적이지만,
문서 구조에 따라 분할된 Chunk가 토큰 제한을 초과하는 경우가 발생할 수 있습니다.

반면, RecursiveCharacterTextSplitter는
줄바꿈, 문장 구분자, 구두점 등 여러 구분자를 순차적으로 적용하며
텍스트를 재귀적으로 분할합니다.

이 방식은 토큰 제한을 안정적으로 만족시키는 데 유리하지만,
분할 과정에서 의미적으로 완전하지 않은 문장 단위로 잘릴 수 있다는 단점이 있습니다.  

단순한 구조의 문서나,
문단 구성이 명확한 텍스트의 경우에는 CharacterTextSplitter로도 충분합니다.

하지만 실제 서비스 환경에서는
문서 길이와 구조가 제각각인 경우가 많기 때문에,
대부분의 RAG 시스템에서는 RecursiveCharacterTextSplitter를 기본 선택지로 사용합니다.

이는 Chunk 크기를 안정적으로 제어하면서도
검색 실패를 줄이는 데 유리하기 때문입니다.

In [24]:
with open("/content/state_of_the_union.txt") as f:
    text = f.read()

In [25]:
#len은 어떤 기준으로 chunk size를 잴 것인가?의 기준이 되는 함수입니다.
#chunk_overlap은 chunk의 앞뒤로 다른 chunk와 설정한 size까지 겹칠 수 있도록 설정하는 것입니다.
text_splitter = CharacterTextSplitter(separator="\n\n", chunk_size=1000, chunk_overlap=100, length_function = len,)
chunks = text_splitter.split_text(text)

Chunk의 내용을 확인해보겠습니다

In [26]:
print(chunks[0])

Madam Speaker, Madam Vice President, our First Lady and Second Gentleman. Members of Congress and the Cabinet. Justices of the Supreme Court. My fellow Americans.  

Last year COVID-19 kept us apart. This year we are finally together again. 

Tonight, we meet as Democrats Republicans and Independents. But most importantly as Americans. 

With a duty to one another to the American people to the Constitution. 

And with an unwavering resolve that freedom will always triumph over tyranny. 

Six days ago, Russia’s Vladimir Putin sought to shake the foundations of the free world thinking he could make it bend to his menacing ways. But he badly miscalculated. 

He thought he could roll into Ukraine and the world would roll over. Instead he met a wall of strength he never imagined. 

He met the Ukrainian people. 

From President Zelenskyy to every Ukrainian, their fearlessness, their courage, their determination, inspires the world.


각 chunk의 길이를 확인해보겠습니다,

In [27]:
length = []
for chunk in chunks:
    length.append(len(chunk))

print(length)

[939, 995, 810, 962, 994, 883, 957, 952, 928, 915, 993, 702, 900, 950, 957, 958, 967, 996, 796, 866, 888, 966, 964, 977, 998, 948, 925, 924, 989, 965, 938, 936, 981, 965, 771, 982, 972, 977, 984, 999, 968, 801]


### 토큰 단위로 텍스트 분할해보기  
  
LLM은 문장을 단어가 아닌 토큰(token) 단위로 처리합니다.
따라서 사람이 인식하는 단어 길이나 문자 수는
실제 모델이 처리하는 입력 길이와 정확히 일치하지 않을 수 있습니다.

이로 인해 문자 수나 단어 수를 기준으로 텍스트를 분할할 경우,
모델의 입력 토큰 제한을 초과하거나
예상보다 훨씬 짧은 문맥만 전달되는 문제가 발생할 수 있습니다.

실제 서비스 환경에서는 이러한 문제를 방지하기 위해,
토큰 단위를 기준으로 텍스트를 분할하는 방식을 사용합니다.
이제 토큰 기준으로 텍스트를 분할해보겠습니다.

In [28]:
!pip install tiktoken

In [29]:
import tiktoken
tokenizer = tiktoken.get_encoding("cl100k_base")

def tiktoken_len(text):
    tokens = tokenizer.encode(
        text
    )
    return len(tokens)

In [30]:
tiktoken_length = []
for chunk in chunks:
    tiktoken_length.append(tiktoken_len(chunk))

print(length)
print(tiktoken_length)

[939, 995, 810, 962, 994, 883, 957, 952, 928, 915, 993, 702, 900, 950, 957, 958, 967, 996, 796, 866, 888, 966, 964, 977, 998, 948, 925, 924, 989, 965, 938, 936, 981, 965, 771, 982, 972, 977, 984, 999, 968, 801]
[197, 198, 163, 190, 203, 182, 195, 197, 206, 205, 218, 148, 188, 205, 216, 215, 209, 224, 176, 187, 201, 197, 201, 215, 222, 202, 203, 204, 229, 206, 184, 204, 197, 194, 156, 200, 194, 221, 203, 225, 209, 187]


글자 수와 토큰 수의 차이를 확인할 수 있습니다 !

### Step3 : TextEmbedding 사용해보기  
Embedding은 텍스트를 컴퓨터가 계산할 수 있는 수치 벡터(vector) 형태로 변환하는 과정입니다.
이 벡터는 문장의 표면적인 형태가 아니라, 의미적 유사성을 반영하도록 설계되어 있습니다.

변환된 벡터는
VectorStore에 저장되거나,
새로운 질의(Query) 벡터와의 유사도 계산을 통해
의미적으로 가까운 문서를 검색하는 데 사용됩니다.

이러한 변환은 대규모 말뭉치로 사전 학습된
Embedding 전용 모델을 통해 이루어지며,
RAG 시스템에서 Retrieval 성능을 결정하는 핵심 요소입니다.

이번 실습에서는
OpenAI 임베딩 모델을 사용해
텍스트를 벡터로 변환해보겠습니다.

In [31]:
import openai

genai 라이브러리의 list_models 함수를 사용하여 사용 가능한 모델들의 목록을 가져옵니다.

In [32]:
client = openai.OpenAI()

In [33]:
for model in client.models.list():
    if "embedding" in model.id:
        print(model.id)

text-embedding-ada-002
text-embedding-3-small
text-embedding-3-large


text-embedding-3-small은 가성비가 좋고, text-embedding-3-large는 성능이 더 강력합니다.

In [34]:
from langchain_openai import OpenAIEmbeddings


embedding_model = OpenAIEmbeddings(model="text-embedding-3-small")

만약 여러분들이 Gemini를 사용하여 구축중이시라면, embedding은 지역에 따라 사용이 제한됩니다.  
주로 유럽권에서 제한되기 때문에, 다음 에러를 확인하신다면 Colab 파일의 서버 저장 위치를 확인 후, 다른 임베딩 모델로 변경해야합니다.  

Error embedding content: 400 User location is not supported for the API use.


In [35]:
#!curl ipinfo.io

In [36]:
# 400 User location is not supported for the API use 오류가 발생한다면, 이 블록을 대신 실행해주세요

# ! pip install -q sentence_transformers

#from langchain.embeddings import HuggingFaceEmbeddings
#embedding_model = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

embedding model 변수에 OpenAI 임베딩모델 혹은 huggingface의 임베딩모델이 할당되었을 것입니다.  
embed_documents 멤버 함수를 사용하여 새 문장을 변환해보겠습니다  

In [37]:
embeddings = embedding_model.embed_documents(
    [
        "This is red apple.",
        "This is yellow banana.",
        "This is green lime.",
    ]
)

임베딩으로 잘 변환되었는지 확인해보겠습니다  

In [38]:
print(embeddings[1])

[0.006435394287109375, -0.0208587646484375, -0.0216064453125, 0.0235595703125, -0.0428466796875, -0.004608154296875, 0.03973388671875, 0.05267333984375, -0.0018892288208007812, -0.0237579345703125, 0.0137939453125, 0.0159912109375, -0.044464111328125, -0.00013935565948486328, -0.007648468017578125, 0.02557373046875, 0.0254058837890625, 0.035491943359375, -0.0517578125, 0.01232147216796875, 0.0248260498046875, -0.00045800209045410156, 0.0193939208984375, 0.07525634765625, -0.0020503997802734375, -0.00921630859375, 0.016815185546875, -0.007152557373046875, 0.02655029296875, -0.048492431640625, 0.043182373046875, -0.04254150390625, 0.01165008544921875, -0.032867431640625, -0.0333251953125, -0.046142578125, 0.0019063949584960938, 0.0225677490234375, -0.01812744140625, 0.03253173828125, 0.0293121337890625, 0.011749267578125, -0.01442718505859375, 0.0030574798583984375, 0.0243377685546875, 0.048187255859375, -0.01355743408203125, 0.049346923828125, -0.040618896484375, 0.04400634765625, 0.045

In [39]:
len(embeddings[1])

1536

새로운 쿼리를 넣어, 임베딩끼리 유사도를 계산해보겠습니다

In [40]:
import numpy as np
from numpy import dot
from numpy.linalg import norm
def cos_sim(A, B):
  return dot(A, B)/(norm(A)*norm(B))

In [59]:
query = ["this is red fruit"]

In [60]:
e_query = embedding_model.embed_documents(query)
print(cos_sim(embeddings[0], e_query[0]))
print(cos_sim(embeddings[1], e_query[0]))
print(cos_sim(embeddings[2], e_query[0]))

0.7478342692665315
0.4898954244712594
0.40836637441688617


빨간 사과와 빨간 과일의 유사도가 많이 높게 나왔습니다!  
  
임베딩 모델은 사용 언어나 필요에 따라 다양하게 교체하여 사용할 수 있습니다.  
해당 링크에서 여러 목록을 확인하실 수 있습니다.  
https://python.langchain.com/docs/integrations/text_embedding/

### Step4 : VectorStore 사용해보기
VectorStore는 텍스트를 Embedding 모델을 통해 벡터(vector)로 변환한 뒤, 이를 저장하고 관리하는 저장소입니다.
이 저장소는 단순한 데이터 보관 공간이 아니라,
벡터 간의 유사도를 빠르게 계산하고 탐색하기 위한 인덱싱 구조를 함께 포함하고 있습니다.

문서나 쿼리가 Embedding된 이후에는,
VectorStore를 통해 의미적으로 유사한 벡터를 효율적으로 검색할 수 있으며,
이 과정이 RAG 시스템의 Retrieval 단계를 담당하게 됩니다.

대표적인 VectorStore로는
Chroma, FAISS 등이 있으며,
각각 로컬 환경과 대규모 서비스 환경에서 널리 사용됩니다.

이번 실습에서는
구성이 단순하고 로컬 환경에서 바로 사용할 수 있는
ChromaDB를 사용해 VectorStore를 구성해보겠습니다.

In [43]:
!pip install chromadb

In [44]:
!pip install langchain-chroma

In [45]:
from langchain_chroma import Chroma

In [46]:
!pip install --upgrade opentelemetry-api
!pip install --upgrade opentelemetry-sdk

제일 처음에 사용했던, PDF를 다시 사용하도록 합니다!  

In [47]:
# 위에서 사용했던 코드입니다
loader = PyPDFLoader("/content/Demian.pdf")
pages = loader.load_and_split()

text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=0, length_function = tiktoken_len)
docs = text_splitter.split_documents(pages)

In [61]:
!pip show chromadb

Name: chromadb
Version: 1.5.9
Summary: Chroma.
Home-page: https://github.com/chroma-core/chroma
Author: 
Author-email: Jeff Huber <jeff@trychroma.com>, Anton Troynikov <anton@trychroma.com>
License: 
Location: /usr/local/lib/python3.12/dist-packages
Requires: bcrypt, build, grpcio, httpx, importlib-resources, jsonschema, kubernetes, mmh3, numpy, onnxruntime, opentelemetry-api, opentelemetry-exporter-otlp-proto-grpc, opentelemetry-sdk, orjson, overrides, pybase64, pydantic, pydantic-settings, pypika, pyyaml, rich, tenacity, tokenizers, tqdm, typer, typing-extensions, uvicorn
Required-by: langchain-chroma


Chroma에 임베딩 시킵니다  

In [49]:
db = Chroma.from_documents(docs, embedding_model)


이제 쿼리를 날려보겠습니다

In [50]:
query = "how Demian look like?"
docs = db.similarity_search(query)

In [51]:
print(docs[0].page_content)

DEMIAN 
with a feeling of nausea, I noticed Demian's expression. 
He had not thrust himself to the front but stood right 
at the back, looking elc!gant and at ease as usual. His 
glance seemed directed at the horse's head, and again it 
. showed that deep, quiet, almost fanatical yet passionate 
absorption. I could not help staring at him for some 
moments and it was then that I felt aware of a very 
uncanny sensation in my remote consciousness. I saw 
Demian's face and remarked that it was not a boy's face 
but a man's and then I saw, or rather became aware, that 
it was not really the face of a man either; it had some­
thing different about it, almost a feminine element. And 
for the time being his face seemed neither masculine 
nor childish, neither old nor young but a hundred years 
old, almost timeless and bearing the mark of other 
periods of history than our own. Animals might look 
thus, trees or stars. I did not know then, of course, I 
did not feel exactly what I am writing a

Face, features, looks like 등 데미안의 생김새를 담고 있는 페이지가 출력되었습니다  
굉장히 빠른 속도로 검색했습니다!  

### Step5 : Retriever 사용해보기  

Retriever는 사용자의 질문을 Embedding 모델을 통해 벡터로 변환한 뒤,
VectorStore에 저장된 문서 벡터들과 비교하여
의미적으로 가장 유사한 문서(Chunk)를 찾아 반환하는 역할을 합니다.

즉, Retriever는
RAG 시스템에서 “어떤 정보를 LLM에게 참고 자료로 줄 것인가”를 결정하는 핵심 컴포넌트이며,
검색 결과의 품질이 곧 최종 답변의 품질로 이어집니다.
  

In [52]:
!pip install -U langchain langchain-classic

In [53]:
from langchain_classic.chains.retrieval_qa.base import RetrievalQA


긴 문서 전체를 한 번에 LLM에 전달하는 대신,
Retriever와 LLM을 결합한 RetrievalQA 체인을 사용하여
문서에서 질문과 관련된 부분만 검색하고,
그 결과를 바탕으로 답변을 생성합니다.

이를 통해 길이가 긴 문서에서도
토큰 제한을 넘지 않으면서, 근거 기반의 질의응답을 수행할 수 있습니다.

In [54]:
from langchain_core.callbacks.streaming_stdout import StreamingStdOutCallbackHandler

# OpenAI 모델로 변경
# streaming=True와 callbacks 설정을 통해 실시간 출력을 활성화합니다.
llm = ChatOpenAI(
    model="gpt-4o",              # 또는 "gpt-4o-mini"
    temperature=0.0,
    streaming=True,              # 실시간 출력을 켭니다
    callbacks=[StreamingStdOutCallbackHandler()], # 출력을 콘솔에 바로 뿌려줍니다
)

체인의 종류와 검색(Retrieval) 방식,
그리고 그에 따른 주요 파라미터를 설정합니다.

이 단계에서는
Retriever가 어떤 전략으로 문서를 검색할지,
그리고 몇 개의 문서를 LLM에게 전달할지를 결정하게 됩니다.
이 선택은 최종 답변의 품질과 직접적으로 연결됩니다.

예를 들어,
MMR(Maximal Marginal Relevance) 방식은
쿼리와의 유사도뿐만 아니라 문서 간의 중복을 줄이고 다양성을 확보하는 재정렬(Re-ranking) 전략입니다.

실무 환경에서는 단일 문서에 정보가 몰리는 것을 방지하고,
LLM이 보다 풍부한 문맥을 참고하도록 하기 위해
MMR 방식이 자주 사용됩니다.

In [55]:
qa = RetrievalQA.from_chain_type(llm, chain_type="stuff",
                                 retriever=db.as_retriever(
                                     search_type="mmr",
                                     search_kwargs={"k": 3, "fetch_k" : 10}),
                                 return_source_documents=True)

위 코드에서 짚고 넘어갈 파라미터는 다음과 같습니다  
🔹 chain_type="stuff"

검색된 문서(Chunk)를 그대로 하나의 Prompt에 모두 삽입하는 방식입니다.
구조가 단순하고 이해하기 쉬워,
RAG 구조를 처음 학습하거나 프로토타입을 만들 때 적합합니다.
단점으로는 문서 수가 많아질 경우
토큰 사용량이 빠르게 증가할 수 있습니다.
실무에서는 초기 검증 단계에서는 stuff를,
문서 수가 많아지면 map_reduce나 refine 방식으로 확장합니다.  

🔹 retriever

VectorStore에서 어떤 문서를 검색할지 결정하는 검색 모듈입니다.
검색 전략과 파라미터 설정에 따라 LLM이 참고하는 정보의 범위와 품질이 달라집니다.  

🔹 search_type="mmr"

MMR(Maximal Marginal Relevance) 검색 방식을 사용합니다. 쿼리와의 유사도뿐만 아니라, 문서 간 중복을 줄여 다양한 문맥을 확보하는 Re-ranking 전략입니다. 실무 환경에서 단일 문서 편향을 줄이기 위해 자주 사용됩니다

🔹 search_kwargs={"k": 3, "fetch_k": 10}  
- fetch_k  
VectorStore에서 우선적으로 가져올 후보 문서 개수입니다. Re-ranking 이전 단계에서 사용됩니다.
- k  
최종적으로 LLM에게 전달할 문서(Chunk)의 개수입니다.

일반적으로 fetch_k > k 로 설정하여 후보 풀을 넉넉히 확보한 뒤, 품질 좋은 문서만 선별하는 방식을 사용합니다.  

🔹 return_source_documents=True

답변 생성에 사용된 원문 문서(Chunk)를 함께 반환합니다. 이를 통해 답변의 출처를 사용자에게 표시하거나 검색 품질을 디버깅하고 RAG 성능을 평가할 수 있습니다. 실무 서비스에서는 거의 필수적으로 사용하는 옵션입니다.

In [56]:
query = "how demian looks like"
result = qa(query)

/tmp/ipykernel_11321/3336337621.py:2: LangChainDeprecationWarning: The method `Chain.__call__` was deprecated in langchain-classic 0.1.0 and will be removed in 2.0.0. Use `invoke` instead.
  result = qa(query)


Demian is described as having a face that is neither distinctly masculine nor childish, neither old nor young, but rather timeless and bearing the mark of other periods of history. His face has an almost feminine element and is different from the rest of the people around him. It is suggested that his appearance is unique, almost like an animal, a spirit, or an image, making him unimaginably different from others.

마크다운 형식으로 출력해봅니다

In [57]:
from IPython.display import Markdown, display
display(Markdown(result["result"]))

Demian is described as having a face that is neither distinctly masculine nor childish, neither old nor young, but rather timeless and bearing the mark of other periods of history. His face has an almost feminine element and is different from the rest of the people around him. It is suggested that his appearance is unique, almost like an animal, a spirit, or an image, making him unimaginably different from others.

RAG를 사용하지 않은 llm 호출도 시도해보세요!

In [58]:
llm2 = ChatOpenAI(
    model="gpt-4o")
request = llm2.invoke("how demian looks like")
display(Markdown(request.content))


If you are referring to the character Emil Sinclair from Hermann Hesse's novel "Demian," he is portrayed as a young man who undergoes significant personal and spiritual growth throughout the story. The book focuses more on his internal development and philosophical musings rather than detailed physical descriptions. However, Sinclair is typically depicted as introspective and sensitive, grappling with complex ideas of self-discovery and duality.

If you are asking about the character Max Demian from the same novel, he is often described as having a striking and enigmatic presence, with an aura of wisdom beyond his years. Demian serves as a mentor and guide to Sinclair, helping him explore deep existential themes.

Hermann Hesse's writing tends to emphasize the inner world of his characters more than their outward appearance, so the physical descriptions are often left to the reader's imagination.

### Quiz
결과의 어떤 부분을 관찰하였을 때, RAG 시스템의 결과를 신뢰할 수 있겠다 생각하셨나요?  

### Answer  
원문에서 답변의 출처를 확인할 수 있었습니다.

## 6. 완성 예제  
앞에서 진행한 내용으로, Demian을 다시 한번 읽어봅시다!  
완성하여 제출해주세요~


필요한 라이브러리를 모두 다운받습니다  

In [63]:
!pip install -U -q langchain langchain-openai langchain-community langchain-core langchain-text-splitters
!pip install -q pypdf
!pip install -q tiktoken
!pip install -q chromadb langchain-chroma
!pip install -U -q langchain-classic

Text splitter 사용을 위한 준비입니다

In [114]:
import os
from google.colab import userdata

os.environ['OPENAI_API_KEY'] = userdata.get('OPENAI_KEY')

### Step 1 Document loader

pdf를 텍스트 데이터 형태로 바꿔주기
Pypdfloader는 페이지 단위로 document만듦



In [128]:
from langchain_community.document_loaders import PyPDFLoader
loader = PyPDFLoader("/content/mad-max-fury-road-2015.pdf")
pages = loader.load_and_split()

In [137]:
print(len(pages))

print(pages[2])

500
page_content='• 
• 
The ROAD WARRIOR sits on a hill in the vast, pitiless wasteland. 
A man UTTERLY ALONE, rocking to and fro ... murmuring to himself. 
This is not meditation. The voice and the movements are too jagged. 
Caption 
As the World Fell ... 
In response to some far off sound, he turns and, for the first time, we see his face ... 
his eyes glisten with crazy-man tears. 
He grabs his gun ... and charges off in what was once a V-8 INTERCEPTOR. The engine 
sounds awful. Clapped out. 
Then, from out of nowhere, PURSUIT VEHICLES are chasing him down ... 
Very soon we hear his engine splutter and die. Then ... an Impact. 
His Car rolls ... once ... twice ... coming to rest right in front of us.' metadata={'producer': '3-Heights(TM) PDF Optimization Shell 4.8.25.2 (http://www.pdf-tools.com)', 'creator': 'tiffps of 23 Jan 95 (EMM - Adobe)', 'creationdate': '2007-03-20T16:07:19-07:00', 'subject': 'Mad Max: Fury Road movie script', 'title': 'Mad Max: Fury Road', 'moddate': '2018-1

In [134]:
print(pages[2].page_content)

• 
• 
The ROAD WARRIOR sits on a hill in the vast, pitiless wasteland. 
A man UTTERLY ALONE, rocking to and fro ... murmuring to himself. 
This is not meditation. The voice and the movements are too jagged. 
Caption 
As the World Fell ... 
In response to some far off sound, he turns and, for the first time, we see his face ... 
his eyes glisten with crazy-man tears. 
He grabs his gun ... and charges off in what was once a V-8 INTERCEPTOR. The engine 
sounds awful. Clapped out. 
Then, from out of nowhere, PURSUIT VEHICLES are chasing him down ... 
Very soon we hear his engine splutter and die. Then ... an Impact. 
His Car rolls ... once ... twice ... coming to rest right in front of us.


### Step 2 Text splitters

토큰 제한에 맞게, 정확도를 높이기 위해 쪼개기

recursivecharactertextsplitters는 여러 기준으로 잘라서 안정적으로 사용함

tiktoken_len 토큰 단위로 세기, 임베딩 모델이랑 가은 방식인 cl100k_base 사용

In [201]:
import tiktoken
from langchain_text_splitters import RecursiveCharacterTextSplitter

tokenizer = tiktoken.get_encoding("cl100k_base")

def tiktoken_len(text):
    tokens = tokenizer.encode(text)
    return len(tokens)
docs = text_splitter.split_documents(pages)

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size = 1000,
    chunk_overlap = 100,
    length_function = len
)



In [202]:
print(len(docs))

504


In [203]:
print(docs[5].page_content)

This high up in the Citadel. .. Is there any escape? 
A massive IRON HOOK is being winched up past him. 
He launches himself...out across the void ... hoping to hitch a ride. 
But he is JERKED up short by the chain around his ankle. 
The WAR BOYS have him. They haul him back into the labyrinth. Despite his ferocity, he is 
no match for their many cudgels and chains. The blows rain down on him and ... gagged once 
again ... he is dragged back into the darkness. 
Title 
•


In [281]:

print(len(docs[0].page_content))

205


### Step 3 Vector Empeddings

텍스트를 숫자 벡터로 변환하기. 관련 문서를 찾아낼 수 있음.

In [206]:
from langchain_openai import OpenAIEmbeddings

embedding_model = OpenAIEmbeddings(model="text-embedding-3-small")

In [239]:
from langchain_chroma import Chroma

db = Chroma.from_documents(docs, embedding_model)


### Step 4 Retrievers

비슷한 상위 내용 중에서 다양한 청크를 선별해서 그 중 k=2개 선별하기

In [258]:
test_query = "Where is Green Place"
test_results = db.similarity_search(test_query, k=2)

for i, doc in enumerate(test_results):
    print((doc.metadata))
    print(doc.page_content[:300])


{'moddate': '2018-10-03T05:14:56+00:00', 'subject': 'Mad Max: Fury Road movie script', 'total_pages': 500, 'producer': '3-Heights(TM) PDF Optimization Shell 4.8.25.2 (http://www.pdf-tools.com)', 'creator': 'tiffps of 23 Jan 95 (EMM - Adobe)', 'creationdate': '2007-03-20T16:07:19-07:00', 'page': 130, 'page_label': '131', 'title': 'Mad Max: Fury Road', 'source': '/content/mad-max-fury-road-2015.pdf'}
• 
• 
• 
The ROAD WARRIOR has found a bandage and is seeing to his mangled fingers. 
ROAD WARRIOR 
So where is this ... Green Place? 
PRAETORIAN 
A long night's run ... head east. 
Straight into your shadow. 
The PRAETORIAN is clipping herself into some kind of body harness. 
PRAETORIAN 
(to the gir
{'producer': '3-Heights(TM) PDF Optimization Shell 4.8.25.2 (http://www.pdf-tools.com)', 'creationdate': '2007-03-20T16:07:19-07:00', 'total_pages': 500, 'subject': 'Mad Max: Fury Road movie script', 'title': 'Mad Max: Fury Road', 'page_label': '331', 'page': 330, 'moddate': '2018-10-03T05:14:56+

### Step 5 Question Answering

In [262]:
from langchain_openai import ChatOpenAI
from langchain_core.callbacks.streaming_stdout import StreamingStdOutCallbackHandler
from langchain_classic.chains.retrieval_qa.base import RetrievalQA

llm = ChatOpenAI(
    model="gpt-4o",
    temperature=0.0,
    streaming=True,
    callbacks=[StreamingStdOutCallbackHandler()]
)

qa = RetrievalQA.from_chain_type(
    llm,
    chain_type="stuff",
    retriever=db.as_retriever(
        search_type="mmr",     # 유사도 + 다양성
        search_kwargs={
            "k": 3,
            "fetch_k": 10      # 10개 중 좋은 3개 선별
        }
    ),
    return_source_documents=True
)



In [268]:
from IPython.display import Markdown, display

query = "where is Green Place?"
print(f"질문: {query}")
result = qa(query)
display(Markdown(result["result"]))

질문: where is Green Place?
The "Green Place" is described as a location that is a long night's run to the east, straight into the shadow. It is characterized by rainfall, vegetation, and a better way to conduct life. However, the exact location is not specified in the provided context.

The "Green Place" is described as a location that is a long night's run to the east, straight into the shadow. It is characterized by rainfall, vegetation, and a better way to conduct life. However, the exact location is not specified in the provided context.

In [271]:
for i, source_doc in enumerate(result["source_documents"]):
    print(f"\n[출처 {i+1}] 페이지: {source_doc.metadata.get('page', '?')}")
    print(source_doc.page_content[:200], "...")


[출처 1] 페이지: 130
• 
• 
• 
The ROAD WARRIOR has found a bandage and is seeing to his mangled fingers. 
ROAD WARRIOR 
So where is this ... Green Place? 
PRAETORIAN 
A long night's run ... head east. 
Straight into your  ...

[출처 2] 페이지: 67
PRAETORIAN 
Angharad! 
SPLENDID 
We're going to the Green Place ... 
The ROAD WARRIOR SHOOTS! 
PRAETORIAN 
No! 
No warning shot.. .just shoots the girl in the leg. 
In the fleshy part around the calf  ...

[출처 3] 페이지: 331
ROAD WARRIOR 
The Many Mothers? 
PRAETORIAN 
The Volvalini. .. (to the girls) ... Try to eat. Drink. 
If you want to piss ... piss in the can. 
The engines will use all the fluid we can spare. 
(to th ...


RAG 없이 그냥 물어본다면

In [280]:
llm2 = ChatOpenAI(model="gpt-4o")
request = llm2.invoke("where is Green Place?")
display(Markdown(request.content))

The term "Green Place" could refer to numerous specific locations or concepts, depending on the context. It might be a street name, the title of a business, an eco-friendly initiative, or even a fictional location in a book or film. Without additional context, it's challenging to pinpoint what specific "Green Place" you are referring to. Could you provide more details or specify the context in which "Green Place" is used?

In [279]:
llm2 = ChatOpenAI(model="gpt-4o")
request = llm2.invoke("where is Green Place in Mad Max?")
display(Markdown(request.content))

In the film "Mad Max: Fury Road," the Green Place is a location that the characters hope to reach as a safe haven, where they believe conditions are more hospitable and sustainable. It is often depicted as a green, fertile area with ample resources, contrasting sharply with the desolate and arid wasteland that makes up much of the film's setting. However, when Furiosa and the group eventually reach the area where the Green Place is supposed to be, they discover that it has become a swamp and is no longer the paradise they hoped for. This realization prompts them to return to the Citadel to reclaim it and make it a new home.